[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kulinskij-tech/Synergetics/blob/main/syn_py_ua/syn_ua_06_reaction_diffusion.ipynb)


# 06. Реакційно-дифузійні системи і структури Тюрінга

_Магістерський курс «Вступ до синергетики» · українська версія_

> **Провідне питання:** Як дифузія може дестабілізувати однорідний стан, стійкий щодо самих хімічних реакцій?

**Як працювати з ноутбуком.** Виконуйте комірки послідовно, згори донизу. Якщо не зазначено інше, усі параметри безрозмірні. Пояснюйте зміст кожної діагностики: сама лише ефектна структура ще не доводить конкретного механізму самоорганізації.


## Результати навчання

Після опрацювання модуля ви зможете:

- вивести дисперсійне співвідношення реакційно-дифузійної системи;
- дослідити стаціонарну точку брюсселятора, поріг Гопфа та нестійкість зі скінченним хвильовим числом;
- сформулювати умови дифузійно індукованої нестійкості для двох компонентів;
- змоделювати двовимірну структуру й відрізнити її морфологію від механізму виникнення.


## Необхідний вступ: швидкості реакцій і дифузія

За законом діючих мас швидкість реакції пропорційна добутку концентрацій реагентів у степенях, заданих стехіометрією. Сума всіх членів утворення і витрати речовин задає локальну кінетику $\dot{\mathbf u}=\mathbf f(\mathbf u)$. Дифузійний потік підкоряється закону Фіка $\mathbf j_i=-D_i\nabla u_i$; локальний закон збереження додає до кожного рівняння член $D_i\nabla^2u_i$.

Для одного поля концентрації дифузія зазвичай згладжує неоднорідності. Проте в системі «активатор—інгібітор» різні коефіцієнти дифузії змінюють зворотний зв'язок, який відчуває просторова збурена мода. Тож питання Тюрінга слід ставити так: чи може матриця $J-k^2D$ мати зростальну моду, хоча $J$ стійка? Граничні умови визначають дозволені просторові моди, тому їх потрібно зазначати завжди.


## Лінійний аналіз нестійкості Тюрінга

Для $\partial_t\mathbf{u}=\mathbf{f}(\mathbf{u})+D\nabla^2\mathbf{u}$ збуримо однорідну стаціонарну точку модою $\delta\mathbf{u}\propto e^{\lambda t+i\mathbf{k}\cdot\mathbf{x}}$. Тоді

$$\det[\lambda I-(J-k^2D)]=0.$$

Нестійкість Тюрінга потребує, щоб реакційна стаціонарна точка була стійкою за $k=0$, але нестійкою в деякій скінченній смузі $k$. Для діагональної матриці $D=\operatorname{diag}(D_u,D_v)$ і $J=\begin{pmatrix}f_u&f_v\\g_u&g_v\end{pmatrix}$ реакційна стійкість вимагає $\operatorname{tr}J<0$ та $\det J>0$. Нерівні коефіцієнти дифузії можуть зробити $\det(J-k^2D)$ від'ємним за скінченного $k$.


## Брюсселятор: мінімальна хімічна дисипативна структура

Брюсселятор — ідеалізована модель за законом діючих мас, запропонована Пригожиним і співробітниками для дослідження хімічного порядку далеко від рівноваги. Це **не** кінетичний механізм реакції Бєлоусова—Жаботинського. Підживлювані компоненти задають параметри $A$ і $B$, а проміжні концентрації $u$ та $v$ задовольняють рівняння

$$\partial_tu=A-(B+1)u+u^2v+D_u\nabla^2u,$$
$$\partial_tv=Bu-u^2v+D_v\nabla^2v.$$

Однорідна стаціонарна точка має координати $(u_*,v_*)=(A,B/A)$, а її реакційний якобіан

$$J_*=\begin{pmatrix}B-1&A^2\\-B&-A^2\end{pmatrix}.$$

Оскільки $\det J_*=A^2>0$, добре перемішаний стан стійкий за $B<1+A^2$ і проходить поріг Гопфа за $B=1+A^2$. За нерівної дифузії він може залишатися стійким при $k=0$, тоді як скінченна смуга просторових мод зростає. Саме ця чиста роздільність робить модель зручною для навчання, хоча її компоненти й параметри не слід буквально ототожнювати з конкретним лабораторним рецептом.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams.update({
    "figure.figsize": (7.2, 4.2),
    "axes.grid": True,
    "grid.alpha": 0.25,
    "font.size": 11,
})
rng = np.random.default_rng(20260904)

# Brusselator dispersion relation and a 1D reaction-diffusion realization.
A, B, Du_b, Dv_b = 1.0, 1.8, 1.0, 10.0  # B < 1 + A**2: reaction-stable
J = np.array([[B-1, A**2], [-B, -A**2]])
k_scan = np.linspace(0, 1.4, 500)
growth = np.array([
    np.max(np.linalg.eigvals(J - kk**2*np.diag([Du_b, Dv_b])).real)
    for kk in k_scan
])
k_pred = k_scan[np.argmax(growth)]

N_b, L_b, dt_b, steps_b = 256, 100.0, 0.002, 40000
dx_b = L_b/N_b
x_b = np.linspace(0, L_b, N_b, endpoint=False)
u_b = A + 0.01*rng.standard_normal(N_b)
v_b = B/A + 0.01*rng.standard_normal(N_b)

def lap_periodic_1d(z):
    return (np.roll(z, 1) - 2*z + np.roll(z, -1))/dx_b**2

for _ in range(steps_b):
    reaction = u_b*u_b*v_b
    du = A - (B+1)*u_b + reaction + Du_b*lap_periodic_1d(u_b)
    dv = B*u_b - reaction + Dv_b*lap_periodic_1d(v_b)
    u_b += dt_b*du
    v_b += dt_b*dv

power_b = np.abs(np.fft.rfft(u_b-u_b.mean()))**2
k_b = 2*np.pi*np.fft.rfftfreq(N_b, d=dx_b)
k_observed = k_b[1:][np.argmax(power_b[1:])]

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].plot(k_scan, growth)
ax[0].axhline(0, color="k", lw=1)
ax[0].axvline(k_pred, color="C1", ls="--", label=fr"fastest $k={k_pred:.2f}$")
ax[0].set(xlabel="wavenumber k", ylabel=r"max Re $\lambda(k)$",
          title="Diffusion destabilizes a finite band")
ax[0].legend()
ax[1].plot(x_b, u_b, label="u")
ax[1].plot(x_b, v_b, label="v")
ax[1].set(xlabel="x", ylabel="concentration",
          title=fr"Brusselator pattern; observed $k={k_observed:.2f}$")
ax[1].legend()
plt.tight_layout()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams.update({
    "figure.figsize": (7.2, 4.2),
    "axes.grid": True,
    "grid.alpha": 0.25,
    "font.size": 11,
})
rng = np.random.default_rng(20260904)

# Gray-Scott reaction-diffusion model with periodic boundaries.
N, steps, dt = 72, 3500, 1.0
Du, Dv, feed, kill = 0.16, 0.08, 0.060, 0.062
u = np.ones((N, N))
v = np.zeros((N, N))
s = slice(N//2-5, N//2+5)
u[s, s] = 0.50
v[s, s] = 0.25
u += 0.01*rng.standard_normal((N, N))
v += 0.01*rng.standard_normal((N, N))

def laplacian(z):
    return (np.roll(z, 1, 0) + np.roll(z, -1, 0)
            + np.roll(z, 1, 1) + np.roll(z, -1, 1) - 4*z)

frames = []
for n in range(steps):
    uvv = u*v*v
    u += dt*(Du*laplacian(u) - uvv + feed*(1-u))
    v += dt*(Dv*laplacian(v) + uvv - (feed+kill)*v)
    if n in [0, 500, 1500, steps-1]:
        frames.append((n, v.copy()))

fig, ax = plt.subplots(1, 4, figsize=(13, 3.2))
for axis, (n, field) in zip(ax, frames):
    image = axis.imshow(field, cmap="magma", origin="lower")
    axis.set_title(f"step {n}")
    axis.set_axis_off()
fig.colorbar(image, ax=ax, shrink=0.75, label="v")
plt.show()


## Межі інтерпретації

Плями Грея—Скотта є повчальним прикладом реакційно-дифузійної структури, однак не кожен набір параметрів цієї моделі відповідає малоамплітудній класичній біфуркації Тюрінга. Механізм потрібно встановлювати за однорідною стаціонарною точкою, якобіаном, спектром із дифузійною поправкою, нелінійним насиченням і протоколом збурення, а не лише за зовнішньою схожістю рисунків.


## Оцінюване завдання

Для брюсселятора виведіть межі Гопфа і Тюрінга та класифікуйте першу нестійкість на площині $(B,D_v/D_u)$ за фіксованого $A$. Оберіть точку, стійку щодо реакцій, але нестійку за Тюрінгом; передбачте смугу нестійких довжин хвиль і порівняйте з моделюванням для двох просторових сіток і двох розмірів області. Зіставте ці докази з морфологією Грея—Скотта нижче.

Подання має містити чітке твердження, його математичне або фізичне обґрунтування, числові свідчення та принаймні одне обмеження висновку.


## Література та атрибуція

**Основне читання з локальної бібліотеки:** Mikhailov, *Foundations of Synergetics I: Distributed Active Systems*; Haken, розділи про хімічні та біологічні системи.

**Перевірені вебджерела:**
- [Brusselator: надана користувачем оглядова сторінка](https://en.wikipedia.org/wiki/Brusselator)
- [Стаття Пригожина—Лефевра про порушення симетрії](https://doi.org/10.1063/1.1668896)
- [NIST: реакційно-дифузійний брюсселятор](https://math.nist.gov/MatrixMarket/data/NEP/brussel/brussel.html)
- [Стаття Тюрінга про морфогенез](https://doi.org/10.1098/rstb.1952.0012)
- [Scholarpedia: reaction-diffusion systems](https://www.scholarpedia.org/article/Reaction-diffusion_systems)

Локальні книги є наданими матеріалами курсу. Вебпосилання мають додатковий характер; їх перевірено 04.09.2026.
